# MoE LiDAR Detection — Router Training & Evaluation

This notebook trains and evaluates the router models at the core of this
project's output-level Mixture-of-Experts pipeline: five frozen, pretrained
LiDAR 3D detectors (CenterPoint-Voxel, CenterPoint-Pillar, PointPillars, SSN,
BEVFusion-LiDAR) are fused by a learned per-class scorer. No expert is
fine-tuned here — this notebook trains only the router.

**What this notebook does, in order:**

1. Load the pre-built router training dataset (`training_data/router_data/`).
2. Train a per-class XGBoost router (bicycle additionally sigmoid-calibrated).
3. Train a per-class FT-Transformer router.


**Before running:** download `train.csv` and `eval.csv` from the Google
Drive link in [`training_data/README.md`](../training_data/README.md) and
place them at `training_data/router_data/`. Steps 1–4 need nothing else.
Step 5 additionally needs the expert prediction JSONs
([`predictions/README.md`](../predictions/README.md)) and a nuScenes
metadata download — see that section for details before running it.


## 0. Setup

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(REPO))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.moe.features import FEATURE_NAMES
from src.moe.xgboost_router import (
    NUSCENES_CLASSES,
    train_xgboost_per_class_routers,
    save_xgboost_per_class_routers,
    load_xgboost_per_class_routers,
)
from src.moe.nn_router import train_nn_per_class_routers, save_nn_per_class_routers

pd.set_option("display.width", 120)
print(f"Repo root: {REPO}")
print(f"Feature vector ({len(FEATURE_NAMES)} features): {FEATURE_NAMES}")


Repo root: /home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection
Feature vector (17 features): ['expert_id', 'detection_score', 'dist_from_ego', 'box_width', 'box_length', 'box_height', 'vel_magnitude', 'n_peer_overlaps', 'max_peer_iou', 'mean_peer_score', 'score_variance', 'expert_agreement', 'n_spatial_overlaps', 'class_agreement', 'max_class_score', 'n_active_experts', 'dist_to_drivable_area']


## 1. Load the router training dataset

`train.csv` (3,411 keyframes / 85 scenes) is used to fit both routers.
`eval.csv` (804 keyframes / 20 scenes) is a disjoint, scene-grouped
calibration split used only for evaluation in this notebook — never for
fitting model parameters. See `configs/moe_final.yaml` → `split` for how
these partitions were built.


In [2]:
DATA_DIR = REPO / "training_data" / "router_data"
train_df = pd.read_csv(DATA_DIR / "train.csv")
eval_df = pd.read_csv(DATA_DIR / "eval.csv")

print(f"train.csv: {len(train_df):,} rows across {train_df['sample_token'].nunique():,} keyframes")
print(f"eval.csv:  {len(eval_df):,} rows across {eval_df['sample_token'].nunique():,} keyframes")
train_df.head()


train.csv: 1,723,052 rows across 3,411 keyframes
eval.csv:  371,102 rows across 804 keyframes


,expert_id,class_id,detection_score,dist_from_ego,box_width,box_length,box_height,vel_magnitude,n_peer_overlaps,max_peer_iou,...,score_variance,expert_agreement,n_spatial_overlaps,class_agreement,max_class_score,n_active_experts,dist_to_drivable_area,label,sample_token,model_name
0,6.0,0.0,0.789173,975.066858,1.874180,4.396540,1.627261,0.0,8.0,0.951364,...,0.112497,1.0,8.0,0.625000,0.92454,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar
1,6.0,0.0,0.687098,953.779420,1.823229,4.433558,1.545039,0.0,7.0,0.957836,...,0.092258,1.0,7.0,0.571429,0.92454,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar
2,6.0,0.0,0.705819,967.571580,1.873642,4.500848,1.521493,0.0,5.0,0.974127,...,0.102421,1.0,5.0,0.800000,0.92454,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar
3,6.0,8.0,0.421172,959.523759,0.350162,0.350411,0.794157,0.0,8.0,0.816722,...,0.021778,1.0,8.0,0.625000,0.67420,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar
4,6.0,8.0,0.434399,965.557422,0.336308,0.330591,0.725230,0.0,4.0,0.707611,...,0.033106,1.0,4.0,1.000000,0.67420,5.0,0.0,1,0046092508b14f40a86760d11f9896bb,bevfusion_lidar


## 2. Train per-class XGBoost routers

One `XGBClassifier` per nuScenes class (200 rounds, depth 6, learning rate
0.05, `scale_pos_weight` for imbalance). Bicycle — the class with the most
extreme imbalance (~2% positive) — is additionally wrapped in
`CalibratedClassifierCV` (sigmoid/Platt scaling) to correct probability-scale
distortion that `scale_pos_weight` introduces; see the module docstring in
`src/moe/xgboost_router.py` for why this matters only for that one class.

Training runs on GPU if available; the fitted models are always reset to
CPU before saving, since this pipeline's inference-time batches are small
enough that GPU prediction is *slower* due to a mismatched-device fallback.


In [3]:
xgb_routers = train_xgboost_per_class_routers(
    train_df, val_df=eval_df, feature_names=FEATURE_NAMES, use_gpu=True,
)

XGB_OUT = REPO / "model_weights" / "router_xgboost"
save_xgboost_per_class_routers(xgb_routers, XGB_OUT)


2026-07-27 19:59:57 | INFO     | src.moe.xgboost_router | Training XGBoost router per class on cuda | features=17


2026-07-27 19:59:59 | INFO     | src.moe.xgboost_router | Trained car                   : 363530 rows, 33.03% positive, n_estimators=152


2026-07-27 19:59:59 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9775  AP=0.9637


2026-07-27 20:00:00 | INFO     | src.moe.xgboost_router | Trained truck                 : 141284 rows, 12.67% positive, n_estimators=120


2026-07-27 20:00:00 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9599  AP=0.7825


...

2026-07-27 20:00:04 | INFO     | src.moe.xgboost_router | Trained pedestrian            : 413558 rows, 17.54% positive, n_estimators=75


2026-07-27 20:00:04 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9585  AP=0.7698


2026-07-27 20:00:05 | INFO     | src.moe.xgboost_router | Trained traffic_cone          : 193577 rows, 6.72% positive, n_estimators=32


2026-07-27 20:00:05 | INFO     | src.moe.xgboost_router |   └─ val AUC=0.9562  AP=0.5234


2026-07-27 20:00:05 | INFO     | src.moe.xgboost_router | Saved XGBoost per-class routers for 10 classes -> /home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection/model_weights/router_xgboost


## 3. Train per-class FT-Transformer routers

A Feature Tokenizer + Transformer (Gorishniy et al., 2021): each of the 17
numeric features (all but the map-derived `dist_to_drivable_area`, which is
XGBoost-only) is projected to a 64-dim embedding, a learnable `[CLS]` token
is prepended, and 3 pre-norm Transformer encoder layers attend across the
resulting sequence. Trained with BCE loss weighted by the inverse class
imbalance ratio; outputs calibrated via isotonic regression on a held-out
10% split of the training data. See `src/moe/nn_router.py` for the full
architecture.


In [4]:
from src.moe.features import NN_FEATURE_NAMES
nn_feature_names = NN_FEATURE_NAMES  # excludes dist_to_drivable_area, by name not position

nn_routers = train_nn_per_class_routers(
    train_df, val_df=eval_df, model_type="ft_transformer", feature_names=nn_feature_names,
)

NN_OUT = REPO / "model_weights" / "router_nn"
save_nn_per_class_routers(nn_routers, NN_OUT, meta={"feature_names": nn_feature_names})


2026-07-27 20:00:05 | INFO     | src.moe.nn_router | Training FT_TRANSFORMER router per class on cuda | features=16 | epochs=30 | batch=4096


2026-07-27 20:00:05 | INFO     | src.moe.nn_router | Training car                   : 363530 rows, 33.0% positive


/home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection/src/moe/nn_router.py:193: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


2026-07-27 20:00:47 | INFO     | src.moe.nn_router |   epoch  5/30  train_loss=0.2578  val_loss=0.2508  lr=1.00e-03


2026-07-27 20:01:30 | INFO     | src.moe.nn_router |   epoch 10/30  train_loss=0.2510  val_loss=0.2457  lr=1.00e-03


...

2026-07-27 20:17:42 | INFO     | src.moe.nn_router |   epoch 30/30  train_loss=0.4664  val_loss=0.4926  lr=2.50e-04


2026-07-27 20:17:42 | INFO     | src.moe.nn_router |   Early stopping at epoch 30 (best val_loss=0.4879 at epoch 25)


2026-07-27 20:17:42 | INFO     | src.moe.nn_router |   Isotonic calibration fitted on 19210 samples (12.2% positive)


2026-07-27 20:17:42 | INFO     | src.moe.nn_router |   └─ val AUC=0.9477  AP=0.4158


2026-07-27 20:17:42 | INFO     | src.moe.nn_router | Saved NN per-class routers for 10 classes → /home/santoshaibox/msaai/capstone/code/direct-data-code/moe-lidar-detection/model_weights/router_nn


## 4. Router-level evaluation

AUC and Average Precision per class on `eval.csv`, for the XGBoost router,
the FT-Transformer router, and their per-class convex blend
(`combined = lambda * p_xgboost + (1 - lambda) * p_ft_transformer`, with
`lambda` from `configs/moe_final.yaml` → `ensemble_lambda`). This is a
*candidate-level* metric — precision/recall on individual scored boxes,
before NMS or temporal refinement — not the official nuScenes mAP, which
Step 5 computes.


In [5]:
import yaml
from sklearn.metrics import roc_auc_score, average_precision_score
from src.moe.features import NN_FEATURE_NAMES
_NN_FEATURE_IDX = [FEATURE_NAMES.index(f) for f in NN_FEATURE_NAMES]

cfg = yaml.safe_load((REPO / "configs" / "moe_final.yaml").read_text())
lambdas = cfg["ensemble_lambda"]

_CLASS_TO_ID = {c: i for i, c in enumerate(
    ["car", "truck", "trailer", "bus", "construction_vehicle",
     "bicycle", "motorcycle", "pedestrian", "traffic_cone", "barrier"]
)}

rows = []
for cls in NUSCENES_CLASSES:
    sub = eval_df[eval_df["class_id"] == _CLASS_TO_ID[cls]]
    if len(sub) == 0 or sub["label"].sum() == 0:
        continue
    X = sub[FEATURE_NAMES].values.astype(np.float32)
    y = sub["label"].values.astype(int)

    p_xgb = xgb_routers[cls].predict_proba(X)[:, 1]
    p_nn = nn_routers[cls].predict_proba(X[:, _NN_FEATURE_IDX])[:, 1]
    lam = lambdas[cls]
    p_ens = lam * p_xgb + (1 - lam) * p_nn

    rows.append({
        "class": cls,
        "n_eval": len(sub),
        "pos_pct": 100 * y.mean(),
        "xgb_AP": average_precision_score(y, p_xgb),
        "nn_AP": average_precision_score(y, p_nn),
        "ensemble_AP": average_precision_score(y, p_ens),
        "lambda": lam,
    })

results_df = pd.DataFrame(rows).set_index("class")
results_df.round(4)


,n_eval,pos_pct,xgb_AP,nn_AP,ensemble_AP,lambda
class,,,,,,
car,90082,36.6766,0.9637,0.9583,0.9638,0.900
truck,34858,11.5583,0.7825,0.7688,0.7994,0.800
construction_vehicle,15838,0.3536,0.0788,0.0515,0.0752,0.800
bus,6731,23.8003,0.9755,0.9544,0.9744,0.900
trailer,12309,10.5126,0.7830,0.7690,0.8045,0.900
barrier,29815,3.9309,0.5347,0.4158,0.5301,0.974
motorcycle,19402,7.0663,0.8206,0.8036,0.8196,0.800
bicycle,39828,2.4204,0.6848,0.6415,0.6854,0.800
pedestrian,86217,13.0624,0.7698,0.7872,0.7906,0.300
